# Evidence Research — developer debug notebook

This notebook is for **experimentation and debugging**, not for the jury (the jury uses the Streamlit app). It drives the exact same engine (`evidence_research.runtime.research`) and then opens up what is normally hidden:

- Tavily search / extract output
- screening decisions (including exclusions)
- atomic claim extraction and normalization
- the evidence graph (sources → evidence → claims → hypotheses)
- source lineage and independent-root counting
- inference proposals and verifier verdicts
- the Bayesian confidence updates, step by step
- the per-round agent bids

No business logic lives here. Everything comes from the shared package.

Run `--demo` scenarios need no API keys; a live run needs `NEBIUS_API_KEY` and `TAVILY_API_KEY` (from `.env`).

In [ ]:
import sys, asyncio
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd

from evidence_research.config import Settings, Depth, load_dotenv_if_present
from evidence_research.runtime import research
from evidence_research.observability.events import format_event
from evidence_research.synthesis.serializer import to_markdown

load_dotenv_if_present()
settings = Settings.from_env()
print("Nebius key set:", settings.nebius_api_key is not None, "| Tavily key set:", settings.tavily_api_key is not None)

## 1. Run the engine

Set `DEMO`:
- `"business"` or `"science"` — offline synthetic scenario (scripted model + fake corpus, no keys);
- `None` — live run of `QUESTION` on Token Factory + Tavily.

In [ ]:
DEMO = "business"   # "business" | "science" | None
QUESTION = "Does Nebius have strong product-market fit in AI cloud?"  # used only when DEMO is None
DEPTH = Depth.STANDARD

result = asyncio.run(research(None if DEMO else QUESTION, settings, DEPTH, demo=DEMO,
                              on_event=lambda e: None, persist=False))
g, rep = result.graph, result.report
# The public contract: a substantive answer + one confidence number (everything else is debug metadata).
print("QUESTION TYPE:", rep.question_type)
print("ANSWER:", rep.answer.answer)
print("CONFIDENCE:", f"{rep.answer.confidence:.0%}", "| penalties:", rep.confidence_breakdown["penalties"])
print("—")
print("status:", rep.status.value, "| stop:", result.session.stop_reason.value if result.session.stop_reason else None)
print(rep.stats)

## 2. Live trace — the decisions the agent made, in order

In [ ]:
for e in result.events.events:
    print(format_event(e))

## 3. Tavily search / extract and screening decisions

In [ ]:
pd.DataFrame([{ 'id': s.id, 'found_by': s.search_origin, 'type': (g.assessments.get(s.id) or None) and g.assessments[s.id].source_type,
                'screening': g.screening[s.id].decision if s.id in g.screening else '',
                'reasons': '; '.join(g.screening[s.id].reasons) if s.id in g.screening else '',
                'extracted': s.extracted, 'chars': len(s.raw_text), 'url': s.url }
              for s in g.sources.values()])

## 4. Atomic claims and their Bayesian confidence (importance never enters truth)

In [ ]:
pd.DataFrame([{ 'id': c.id, 'kind': c.kind, 'claim': c.natural_language[:80], 'posterior': round(c.truth_probability, 3),
                'sufficiency': round(c.evidence_sufficiency, 3), 'importance': c.importance, 'status': c.status.value,
                'roots': len(g.roots_for_claim(c.id)), 'evidence': len(g.evidence_for(c.id)) }
              for c in g.claims.values() if c.kind == 'fact']).sort_values('importance', ascending=False).head(20)

### Step-by-step Bayesian update for one claim
Pick a contested or important claim and watch each evidence item move the odds (cluster-aware).

In [ ]:
from evidence_research.probability.bayesian import update_claim
from evidence_research.probability.engine import evidence_quality

cid = max((c.id for c in g.claims.values() if c.kind == 'fact' and g.evidence_for(c.id)),
          key=lambda i: len(g.evidence_for(i)))
print('claim:', g.claims[cid].natural_language, '\n')
items = []
for e in g.evidence_for(cid):
    q = evidence_quality(g, e, g.claims[cid], settings)
    items.append((e, q.strength, g.root_of.get(e.source_id, e.source_id)))
    print(f"{e.stance:<11} strength={q.strength:.2f} LR={q.likelihood_ratio:<5} root={g.root_of.get(e.source_id, e.source_id)}  \u201c{e.exact_excerpt[:80]}\u201d")
res = update_claim(settings.heuristics.claim_prior, items, settings.heuristics)
print('\nprior', res.prior, '-> posterior', res.posterior, '| independent roots', res.independent_roots)
for c in res.contributions:
    print(f"  {c.stance:<11} weight={c.weight} log_lr={c.log_lr:+.3f} ({'primary' if c.weight==1 else 'discounted derivative'})")

## 5. Source lineage — URLs → publication families → independent roots

In [ ]:
for e in g.lineage:
    print(f"{e.child_source_id} --{e.kind}--> {e.parent_source_id}  p={e.dependency_probability}  ({'; '.join(e.evidence)})")
used = {ev.source_id for ev in g.evidence.values() if ev.verification_status == 'verified'}
print(f"\n{len(used)} URLs with evidence -> {rep.publication_family_count} families -> {rep.independent_root_count} independent roots")

## 6. Inferences — model proposes, verifier checks (and the logic rules R1–R3)

In [ ]:
pd.DataFrame([{ 'id': i.id, 'type': i.inference_type, 'dir': i.direction, 'premises': ', '.join(i.premise_ids),
                'conclusion': i.conclusion_claim_id, 'status': i.status, 'score': i.verifier_score,
                'statement': i.statement[:70], 'why_not': '; '.join(i.failure_reasons)[:80] }
              for i in g.inferences.values()])

## 7. Hypotheses, research gaps, and the per-round agent bids

In [ ]:
display(pd.DataFrame([{ 'id': h.id, 'posterior': h.posterior, 'sufficiency': h.sufficiency, 'status': h.status, 'text': h.text[:70] }
                      for h in rep.hypotheses]))
display(pd.DataFrame([{ 'round': it, 'agent': b.agent_type, 'utility': b.utility, 'reason': b.reasons[0] if b.reasons else '' }
                      for it, bids in g.bids for b in bids]))
print('open gaps:')
for gap in rep.gaps:
    print(' ', gap.level, gap.gap_type, gap.description[:90])

## 8. Final report (Minto pyramid) and a live re-check of every citation

In [ ]:
from IPython.display import Markdown
Markdown(to_markdown(rep))

In [ ]:
from evidence_research.verification.citation_audit import audit, summary
if DEMO:
    from evidence_research.demo import FakeSearch, get_scenario
    client = FakeSearch(get_scenario(DEMO).pages)
else:
    from evidence_research.retrieval.tavily_client import TavilyResearchClient
    client = TavilyResearchClient(settings)
checks = asyncio.run(audit(rep.model_dump(mode='json'), client))
print(summary(checks), '\n')
pd.DataFrame([{ 'n': c.citation, 'status': c.status, 'stance': c.stance, 'quote': ' '.join(c.excerpt.split())[:90] } for c in checks])